In [14]:
import re
from datetime import datetime
from typing import Dict, List, Optional
import pandas as pd

def find_and_parse_logs(log_content: str) -> Dict[str, List[Dict]]:
    """
    Find all log entries in a larger log file and parse them by type.
    Returns a dictionary with 'with_split' and 'without_split' keys.
    """
    # Pattern to match the entire log entry
    pattern = r'(={10,}.*?={10,}\s+Rollout start,.*?Duration: [\d.]+ seconds)'
    
    matches = re.findall(pattern, log_content, re.DOTALL)
    
    with_split = []
    without_split = []
    
    for match in matches:
        parsed = parse_log_entry(match)
        if parsed:
            if 'split' in parsed:
                with_split.append(parsed)
            else:
                without_split.append(parsed)
    
    return {
        'with_split': with_split,
        'without_split': without_split
    }


def parse_log_entry(log_text: str) -> Optional[Dict]:
    """
    Parse a single log entry and return a dictionary with extracted information.
    Handles both formats (with and without Split).
    """
    # Parse the header line (between ===)
    header_match = re.search(
        r'=+ Iter (\d+)(?: Split (\d+))? Batch (\d+) TP (\d+) Deterministic (\w+) =+',
        log_text
    )
    
    if not header_match:
        return None
    
    result = {
        'iter': int(header_match.group(1)),
        'batch': int(header_match.group(3)),
        'tp': int(header_match.group(4)),
        'deterministic': header_match.group(5) == 'True'
    }
    
    # Add split if it exists
    if header_match.group(2) is not None:
        result['split'] = int(header_match.group(2))
    
    # Parse start time
    start_match = re.search(r'Rollout start, start_time=(.+)', log_text)
    if start_match:
        result['start_time'] = datetime.fromisoformat(start_match.group(1).strip())
    
    # Parse end time
    end_match = re.search(r'Rollout end, end_time=(.+)', log_text)
    if end_match:
        result['end_time'] = datetime.fromisoformat(end_match.group(1).strip())
    
    # Parse duration (seconds)
    duration_match = re.search(r'Duration: ([\d.]+) seconds', log_text)
    if duration_match:
        result['duration_seconds'] = float(duration_match.group(1))
    
    # Parse rollout time (timedelta string)
    rollout_time_match = re.search(r'Rollout time, end_time - start_time=(.+)', log_text)
    if rollout_time_match:
        result['rollout_time'] = rollout_time_match.group(1).strip()
    
    return result



In [15]:
with open('dist.log', 'r') as f:
    full_log = f.read()

# Parse the entire log file
results = find_and_parse_logs(full_log)

In [19]:
print("=" * 60)
print(f"Found {len(results['with_split'])} entries WITH split:")
print("=" * 60)
df = pd.DataFrame(results['with_split'])
df.to_csv('with_split.csv', index=False)


Found 90 entries WITH split:


In [20]:
print("=" * 60)
print(f"Found {len(results['without_split'])} entries WITHOUT split:")
print("=" * 60)
df = pd.DataFrame(results['without_split'])
df.to_csv('without_split.csv', index=False)


Found 80 entries WITHOUT split:


In [ ]:

# You can also access individual entries
print("=" * 60)
print("Detailed view of first entry with split:")
print("=" * 60)
if results['with_split']:
    import pprint
    pprint.pprint(results['with_split'][0])